**STATUS:** ALIVE (descriptive)
**LAST-AUDIT:** 2026-05-05
**FEEDS:** B12, B13, B14 (post-pivot headline bid-shape evidence); B8 (cross-validation with the IB-canonical study in nb09)
**CLAIM:** Comprehensive descriptive atlas of OMIE bid behaviour across regimes, technologies, firms, plants, hours of day, and seasons, covering both auction markets (Day-Ahead and Intraday Auctions) symmetrically.


# Bid-shape atlas — both auction markets, every dimension

Workshop feedback (May 5 CEMFI): bid-shape is the most important piece of the project. The within-market granularity mechanism predicts that operators bid differently across the four quarters of an hour in critical (steep within-hour profile) but not in flat (constant within-hour profile) hours, and that this should be visible in the SHAPE of bids: more tranches, steeper ladder, lower average price in critical hours.

This notebook covers **both auction markets symmetrically**:

- **Part A — Day-Ahead market (DA)** — DET joined to CAB. Restricted to post-2025-03-19 because pre-MTU15-IDA DA bid-prices are 0-padded due to a parser artefact. Three regimes: DA60/ID15-prebo, DA60/ID15-reforz, DA15/ID15.
- **Part B — Intraday Auctions market (IDA)** — IDET joined to ICAB. Spans the full reform sequence cleanly. Six regimes: pre-DA15, 3-sess, ISP15-win, DA60/ID15-prebo, DA60/ID15-reforz, DA15/ID15.

For each market, the same eight analyses run with identical code (driven by shared helper functions):

1. **Across regimes** (overall).
2. **By technology** (CCGT, hydro, nuclear, wind, solar) × regime.
3. **By firm** (dominant vs fringe + per-dominant-firm) × regime.
4. **Hour-of-day** × regime heatmap.
5. **Critical-flat differential** by tech × regime — the headline mechanism.
6. **Seasonality** — month × regime.
7. **Plant-level drilldown** — top units by volume.
8. **Within-hour granularity exploitation** — post-MTU15 panel only; mechanical-repeat indicator.

Bid-shape measures throughout (per offer = per (date, period, unit) + session for IDA):

- `n_tranches` — distinct price tranches.
- `q_total` — total quantity offered.
- `p_min`, `p_max`, `p_avg` — min, max, qty-weighted mean price.
- `p_range = p_max − p_min`.
- `slope = (p_max − p_min) / q_total`.


In [ ]:
# Setup
from __future__ import annotations
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path('/Users/pabloparamio/Desktop/CEMFI/2nd Year/Master Thesis/mtu15_project')
DET   = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_diario' / 'ofertas' / 'det_all.parquet'
CAB   = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_diario' / 'ofertas' / 'cab_all.parquet'
IDET  = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_intradiario_subastas' / 'ofertas' / 'idet_all.parquet'
ICAB  = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_intradiario_subastas' / 'ofertas' / 'icab_all.parquet'
PDBCE = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_diario' / 'programas' / 'pdbce_all.parquet'
LISTA = PROJECT / 'data' / 'external' / 'omie_reference' / 'lista_unidades.csv'
OUT   = PROJECT / 'figures' / 'working'
OUT.mkdir(exist_ok=True, parents=True)

# Reform regime cuts
PRE_IDA_END    = pd.Timestamp('2024-06-14')
THREE_SESS_END = pd.Timestamp('2024-12-01')
ISP15_END      = pd.Timestamp('2025-03-19')   # MTU15-IDA
BLACKOUT       = pd.Timestamp('2025-04-28')
MTU15_DA_DATE  = pd.Timestamp('2025-10-01')   # MTU15-DA

CRITICAL_HOURS = [7, 8, 16, 17, 18]
FLAT_HOURS     = [3, 4, 5]
DOMINANT       = ['IB', 'GE', 'GN', 'HC']
TECHS_PRIMARY  = ['CCGT', 'hydro', 'nuclear', 'wind', 'solar']

REGIMES_6     = ['pre-DA15', '3-sess', 'ISP15-win', 'DA60/ID15-prebo', 'DA60/ID15-reforz', 'DA15/ID15']
REGIMES_3_DA  = ['DA60/ID15-prebo', 'DA60/ID15-reforz', 'DA15/ID15']

def regime_of(d):
    d = pd.Timestamp(d)
    if d < PRE_IDA_END:    return 'pre-DA15'
    if d < THREE_SESS_END: return '3-sess'
    if d < ISP15_END:      return 'ISP15-win'
    if d < BLACKOUT:       return 'DA60/ID15-prebo'
    if d < MTU15_DA_DATE:  return 'DA60/ID15-reforz'
    return 'DA15/ID15'

con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")
print('OK — paths and constants set.')


In [ ]:
# Firm + tech mapping per unit_code

# 1. firm from PDBCE grupo_empresarial (thermal/hydro/nuclear): latest per unit
firms_thermal = con.execute(f'''
    SELECT unit_code, grupo_empresarial AS firm FROM (
      SELECT unit_code, grupo_empresarial,
             ROW_NUMBER() OVER (PARTITION BY unit_code ORDER BY date DESC) AS rn
      FROM '{PDBCE}' WHERE grupo_empresarial IS NOT NULL) WHERE rn = 1
''').df()

# 2. tech + owner_agent from LISTA_UNIDADES
units = pd.read_csv(LISTA)[['unit_code', 'technology', 'owner_agent']]

def tech_bucket(t):
    if t is None or pd.isna(t): return 'unknown'
    t = str(t)
    if 'Solar' in t: return 'solar'
    if 'Eólica' in t: return 'wind'
    if 'Hidráulica' in t or 'Hidraulic' in t: return 'hydro'
    if 'Ciclo Combinado' in t: return 'CCGT'
    if 'Nuclear' in t: return 'nuclear'
    if 'Térmica no Renovab' in t: return 'thermal_nonRE'
    if 'Térmica Renovable' in t: return 'thermal_RE'
    if 'Almacen' in t: return 'storage'
    if 'Comerc' in t or 'Compra' in t or 'Consumi' in t or 'Import' in t: return 'comm/buyer'
    return 'other'

def owner_to_firm(s):
    if s is None or pd.isna(s): return None
    s = str(s).upper()
    if 'IBERDROLA' in s: return 'IB'
    if 'ENDESA' in s or 'ENEL GREEN POWER' in s: return 'GE'
    if 'NATURGY' in s or 'GAS NATURAL' in s: return 'GN'
    if 'EDP' in s or 'HIDROELÉCTRICA DEL CANTÁBRICO' in s or 'HC ENERG' in s: return 'HC'
    return None

units['tech'] = units['technology'].apply(tech_bucket)
units['firm_from_owner'] = units['owner_agent'].apply(owner_to_firm)

m = units.merge(firms_thermal, on='unit_code', how='left')
m['firm'] = m['firm'].fillna(m['firm_from_owner'])
m['firm_class'] = np.where(m['firm'].isin(DOMINANT), 'dominant', 'fringe')

unit_map = m[['unit_code', 'firm', 'tech', 'firm_class']].dropna(subset=['firm'])
con.register('unit_map', unit_map)

print(f'unit_map rows: {len(unit_map):,}')
print(unit_map.groupby(['firm_class','tech']).size().unstack(fill_value=0))


In [ ]:
# Analysis helper functions — used identically for both markets

def regime_agg(df, group_cols):
    g = df.groupby(group_cols)
    return pd.DataFrame({
        'n_offers':   g.size(),
        'tranches_med':  g['n_tranches'].median(),
        'tranches_p90':  g['n_tranches'].quantile(0.90),
        'q_total_med':   g['q_total'].median(),
        'p_avg_med':     g['p_avg'].median(),
        'p_range_med':   g['p_range'].median(),
        'p_range_p90':   g['p_range'].quantile(0.90),
        'slope_med':     g['slope'].median(),
        'slope_p90':     g['slope'].quantile(0.90),
    }).reset_index()

def crit_flat_diff(df, group_cols, measure, fn='median'):
    df = df.copy()
    df['hour_class'] = np.where(df['hour'].isin(CRITICAL_HOURS), 'critical',
                          np.where(df['hour'].isin(FLAT_HOURS), 'flat', 'other'))
    df = df[df['hour_class'].isin(['critical', 'flat'])]
    if fn == 'median':
        agg = df.groupby(group_cols + ['hour_class'])[measure].median().unstack('hour_class')
    elif fn == 'p90':
        agg = df.groupby(group_cols + ['hour_class'])[measure].quantile(0.90).unstack('hour_class')
    agg['diff'] = agg['critical'] - agg['flat']
    return agg.reset_index()

# Color convention: DA = orange, IDA = blue
COLOR = {'DA': 'tab:orange', 'IDA': 'steelblue'}

def section_overall(panel, regimes, market):
    """Across regimes (overall)."""
    summary = regime_agg(panel, ['regime']).set_index('regime').reindex(regimes).reset_index()
    print(f'{market} bid measures by regime:')
    print(summary.round(2).to_string(index=False))
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
    for ax, (col, title) in zip(axes, [
        ('tranches_med', 'tranches per offer (median)'),
        ('p_range_p90',  'price range €/MWh (P90)'),
        ('slope_p90',    'ladder slope (P90)'),
        ('q_total_med',  'q_total MWh (median)'),
    ]):
        ax.bar(summary['regime'], summary[col], color=COLOR[market])
        ax.set_title(title, fontsize=10)
        ax.tick_params(axis='x', rotation=30)
        ax.grid(alpha=0.3)
    plt.suptitle(f'{market} bid measures across regimes (all units, all tech)', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_01_overall.png', dpi=110, bbox_inches='tight')
    plt.show()
    return summary

def section_by_tech(panel, regimes, market):
    """By technology × regime — three measures."""
    tech_reg = regime_agg(panel[panel['tech'].isin(TECHS_PRIMARY)], ['tech', 'regime'])
    def grid(measure, title, log=False, suffix=''):
        fig, axes = plt.subplots(1, 5, figsize=(15, 3.4), sharey=True)
        for ax, t in zip(axes, TECHS_PRIMARY):
            sub = tech_reg[tech_reg['tech'] == t].set_index('regime').reindex(regimes).reset_index()
            ax.bar(sub['regime'], sub[measure], color=COLOR[market])
            ax.set_title(t, fontsize=10)
            ax.tick_params(axis='x', rotation=30, labelsize=8)
            ax.grid(alpha=0.3)
            if log: ax.set_yscale('log')
        plt.suptitle(title, fontsize=11)
        plt.tight_layout()
        plt.savefig(OUT / f'bid_atlas_{market}_02_tech_{suffix}.png', dpi=110, bbox_inches='tight')
        plt.show()
    grid('tranches_med', f'Tranches per offer (median) — {market} bids by tech × regime',
         suffix='tranches')
    grid('p_range_p90',  f'Price range €/MWh (P90) — {market} bids by tech × regime',
         log=True, suffix='prange')
    grid('p_avg_med',    f'Avg price €/MWh (median) — {market} bids by tech × regime',
         suffix='pavg')

def section_by_firm(panel, regimes, market):
    """By firm class (dominant vs fringe) and per-dominant-firm × regime."""
    dom_reg = regime_agg(panel, ['firm_class', 'regime'])
    print(f'{market} bid measures by firm class × regime:')
    print(dom_reg.round(2).to_string(index=False))
    # dominant vs fringe — three measures
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, (col, title) in zip(axes, [
        ('tranches_med', 'tranches per offer (median)'),
        ('p_range_p90',  'price range €/MWh (P90)'),
        ('slope_p90',    'ladder slope (P90)'),
    ]):
        pivot = dom_reg.pivot(index='regime', columns='firm_class', values=col).reindex(regimes)
        pivot.plot(kind='bar', ax=ax, color=['tab:blue', 'tab:orange'])
        ax.set_title(title, fontsize=10)
        ax.tick_params(axis='x', rotation=30)
        ax.grid(alpha=0.3)
        ax.set_xlabel('')
    plt.suptitle(f'{market} bid measures: Dominant vs Fringe across regimes', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_03_dom_vs_fringe.png', dpi=110, bbox_inches='tight')
    plt.show()
    # per dominant firm
    firm_reg = regime_agg(panel[panel['firm_class'] == 'dominant'], ['firm', 'regime'])
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
    for ax, f in zip(axes, DOMINANT):
        sub = firm_reg[firm_reg['firm'] == f].set_index('regime').reindex(regimes).reset_index()
        ax.bar(sub['regime'], sub['tranches_med'], color=COLOR[market])
        ax.set_title(f, fontsize=10)
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        ax.grid(alpha=0.3)
    plt.suptitle(f'Tranches per offer (median) — {market} bids by dominant firm × regime', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_04_dominant_per_firm.png', dpi=110, bbox_inches='tight')
    plt.show()

def section_hour_heatmap(panel, regimes, market):
    """Hour-of-day × regime heatmaps (two measures)."""
    hour_reg = (panel.groupby(['hour', 'regime'])
                     .agg(tranches=('n_tranches', 'median'),
                          prange=('p_range', lambda s: s.quantile(0.90)))
                     .reset_index())
    def heat(val, title, cmap, suffix):
        pivot = hour_reg.pivot(index='hour', columns='regime', values=val).reindex(columns=regimes)
        fig, ax = plt.subplots(figsize=(max(7, 1.4*len(regimes)+3), 5.5))
        im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, origin='lower')
        ax.set_yticks(range(24)); ax.set_yticklabels(range(24), fontsize=8)
        ax.set_xticks(range(len(regimes))); ax.set_xticklabels(regimes, rotation=30, ha='right', fontsize=9)
        ax.set_ylabel('Hour of day')
        for h in CRITICAL_HOURS: ax.axhline(h, color='red',  alpha=0.25, lw=0.6)
        for h in FLAT_HOURS:     ax.axhline(h, color='blue', alpha=0.25, lw=0.6)
        plt.colorbar(im, ax=ax, label=val)
        ax.set_title(f'{title}\nred=critical hours, blue=flat hours', fontsize=10)
        plt.tight_layout()
        plt.savefig(OUT / f'bid_atlas_{market}_05_hour_{suffix}.png', dpi=110, bbox_inches='tight')
        plt.show()
    heat('tranches', f'{market} bid tranches per offer (median)', 'viridis', 'tranches')
    heat('prange',   f'{market} bid price range €/MWh (P90)',     'magma',   'prange')

def section_crit_flat(panel, regimes, market):
    """Critical-flat differential by tech × regime — two measures."""
    cf_t = crit_flat_diff(panel[panel['tech'].isin(TECHS_PRIMARY)], ['tech', 'regime'], 'n_tranches', 'median')
    fig, ax = plt.subplots(figsize=(max(9, 1.6*len(regimes)+2), 4))
    pivot = cf_t.pivot(index='regime', columns='tech', values='diff').reindex(regimes)
    pivot.plot(kind='bar', ax=ax, width=0.85)
    ax.axhline(0, color='black', lw=0.6)
    ax.set_title(f'Critical − flat differential in {market} bid tranches per offer (median), by tech × regime', fontsize=10)
    ax.set_ylabel('crit − flat (median tranches per offer)')
    ax.tick_params(axis='x', rotation=30); ax.legend(title='tech', fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_06_crit_flat_tranches.png', dpi=110, bbox_inches='tight')
    plt.show()
    cf_p = crit_flat_diff(panel[panel['tech'].isin(TECHS_PRIMARY)], ['tech', 'regime'], 'p_range', 'p90')
    fig, ax = plt.subplots(figsize=(max(9, 1.6*len(regimes)+2), 4))
    pivot = cf_p.pivot(index='regime', columns='tech', values='diff').reindex(regimes)
    pivot.plot(kind='bar', ax=ax, width=0.85)
    ax.axhline(0, color='black', lw=0.6)
    ax.set_title(f'Critical − flat differential in {market} price range €/MWh (P90), by tech × regime', fontsize=10)
    ax.set_ylabel('crit − flat (€/MWh, P90)')
    ax.tick_params(axis='x', rotation=30); ax.legend(title='tech', fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_07_crit_flat_prange.png', dpi=110, bbox_inches='tight')
    plt.show()

def section_seasonality(panel, regimes, market):
    """Monthly patterns — heatmap regime × month, for CCGT and hydro."""
    df = panel.copy()
    df['month'] = df['date'].dt.month
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, t in zip(axes, ['CCGT', 'hydro']):
        sub = df[df['tech'] == t]
        panel_p = sub.groupby(['regime', 'month'])['n_tranches'].median().unstack('month').reindex(regimes)
        # Reindex columns to ensure all 12 months appear (NaN for missing)
        panel_p = panel_p.reindex(columns=range(1, 13))
        im = ax.imshow(panel_p.values, aspect='auto', cmap='viridis')
        ax.set_yticks(range(len(panel_p.index))); ax.set_yticklabels(panel_p.index, fontsize=9)
        ax.set_xticks(range(12)); ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
        ax.set_xlabel('Month')
        ax.set_title(f'{t}: {market} tranches per offer (median), by regime × month', fontsize=10)
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_08_seasonality.png', dpi=110, bbox_inches='tight')
    plt.show()

def section_plant_level(panel, regimes, market, top_n=8):
    """Top units by total volume in each tech, per-unit median tranches across regimes."""
    for t in ['CCGT', 'hydro', 'nuclear']:
        top = (panel[panel['tech'] == t]
               .groupby(['unit_code', 'firm'])['q_total'].sum()
               .reset_index().nlargest(top_n, 'q_total'))
        if len(top) == 0:
            continue
        sub = panel[panel['unit_code'].isin(top['unit_code'])]
        panel_p = (sub.groupby(['unit_code', 'regime'])['n_tranches']
                      .median().unstack('regime').reindex(columns=regimes)
                      .reindex(top['unit_code'].values))
        fig, ax = plt.subplots(figsize=(max(9, 1.4*len(regimes)+3), 5))
        im = ax.imshow(panel_p.values, aspect='auto', cmap='viridis')
        ax.set_yticks(range(len(panel_p.index))); ax.set_yticklabels(
            [f'{u} ({top[top.unit_code==u].firm.iloc[0]})' for u in panel_p.index], fontsize=9)
        ax.set_xticks(range(len(regimes))); ax.set_xticklabels(regimes, rotation=30, ha='right', fontsize=9)
        ax.set_title(f'Top-{top_n} {t} units: {market} tranches per offer (median) across regimes', fontsize=10)
        plt.colorbar(im, ax=ax, label='median tranches')
        plt.tight_layout()
        plt.savefig(OUT / f'bid_atlas_{market}_09_top_{t}.png', dpi=110, bbox_inches='tight')
        plt.show()

def section_within_hour(panel, mtu15_cutoff, market):
    """Within-hour granularity exploitation — mechanical-repeat indicator.

    For each (date, hour, unit) post-MTU15 cutoff, check whether all four
    quarter-bids are identical (mechanical repeat) or distinct (granularity exploited)."""
    sub = panel[panel['date'] >= mtu15_cutoff].copy()
    if 'quarter' not in sub.columns:
        sub['quarter'] = (sub['period'] - 1) % 4
    g = sub.groupby(['date', 'hour', 'unit_code'])
    hu = pd.DataFrame({
        'firm':       g['firm'].first(),
        'tech':       g['tech'].first(),
        'firm_class': g['firm_class'].first(),
        'n_quarters': g.size(),
        'tranches_var': g['n_tranches'].nunique(),
        'pavg_var':     g['p_avg'].nunique(),
        'prange_var':   g['p_range'].nunique(),
    }).reset_index()
    hu['hour_class'] = np.where(hu['hour'].isin(CRITICAL_HOURS), 'critical',
                          np.where(hu['hour'].isin(FLAT_HOURS), 'flat', 'other'))
    hu['mechanical_repeat'] = ((hu['tranches_var'] == 1) &
                                (hu['pavg_var'] == 1) &
                                (hu['prange_var'] == 1)).astype(int)
    hu_4 = hu[hu['n_quarters'] == 4]
    rate = (hu_4.groupby(['tech', 'hour_class'])['mechanical_repeat']
                .mean().unstack('hour_class')) * 100
    rate = rate.reindex(columns=['critical', 'flat', 'other'])
    print(f'{market} mechanical-repeat rate (% unit-hours where all 4 quarter-bids are identical), by tech × hour class:')
    print(rate.round(1).to_string())
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_techs = [t for t in TECHS_PRIMARY if t in rate.index]
    rate.loc[plot_techs, ['critical', 'flat']].plot(kind='bar', ax=ax, color=['tab:red', 'tab:blue'])
    ax.set_title(f'{market} mechanical-repeat rate post-MTU15: % unit-hours with identical 4-quarter bids', fontsize=10)
    ax.set_ylabel('% of unit-hours'); ax.set_ylim(0, 100); ax.tick_params(axis='x', rotation=0); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_10_mechanical_repeat.png', dpi=110, bbox_inches='tight')
    plt.show()

print('Helper functions defined.')


In [ ]:
# Helper to plot raw bid curves (price-quantity step functions)

def _fetch_tranches_da(unit_codes, dates):
    units_list = ",".join(f"'{u}'" for u in unit_codes)
    dates_list = ",".join(f"DATE '{pd.Timestamp(d).date()}'" for d in dates)
    return con.execute(f"""
        WITH cab_v AS (
          SELECT CAST(date AS DATE) AS d, offer_code, version, unit_code,
                 ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), offer_code ORDER BY version DESC) AS rn
          FROM '{CAB}'
          WHERE buy_sell='V' AND unit_code IN ({units_list})
            AND CAST(date AS DATE) IN ({dates_list})
        ),
        cab_l AS (SELECT * FROM cab_v WHERE rn = 1)
        SELECT CAST(det.date AS DATE) AS date, det.period, cab_l.unit_code,
               det.price_eur_mwh AS price, det.quantity_mw AS qty
        FROM '{DET}' det
          JOIN cab_l ON CAST(det.date AS DATE) = cab_l.d
                     AND det.offer_code = cab_l.offer_code
                     AND det.version = cab_l.version
        WHERE det.quantity_mw IS NOT NULL AND det.quantity_mw > 0
          AND det.price_eur_mwh IS NOT NULL
          AND CAST(det.date AS DATE) IN ({dates_list})
          AND det.period BETWEEN 1 AND 96
    """).df()

def _fetch_tranches_ida(unit_codes, dates):
    units_list = ",".join(f"'{u}'" for u in unit_codes)
    dates_list = ",".join(f"DATE '{pd.Timestamp(d).date()}'" for d in dates)
    return con.execute(f"""
        WITH icab_v AS (
          SELECT CAST(date AS DATE) AS d, session_number, offer_code, version,
                 ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), session_number, offer_code ORDER BY version DESC) AS rn
          FROM '{ICAB}'
          WHERE buy_sell='V' AND CAST(date AS DATE) IN ({dates_list})
        ),
        icab_l AS (SELECT * FROM icab_v WHERE rn = 1)
        SELECT CAST(idet.date AS DATE) AS date, idet.session_number, idet.period, idet.unit_code,
               idet.price_eur_mwh AS price, idet.quantity_mw AS qty
        FROM '{IDET}' idet
          JOIN icab_l ON CAST(idet.date AS DATE) = icab_l.d
                      AND idet.session_number = icab_l.session_number
                      AND idet.offer_code = icab_l.offer_code
                      AND idet.version = icab_l.version
        WHERE idet.quantity_mw IS NOT NULL AND idet.quantity_mw > 0
          AND idet.price_eur_mwh IS NOT NULL
          AND idet.unit_code IN ({units_list})
          AND CAST(idet.date AS DATE) IN ({dates_list})
          AND idet.period BETWEEN 1 AND 96
    """).df()

def section_bid_curves(panel, regimes, market, n_curves_per_regime=15,
                        focus_regime='DA15/ID15', y_clip=(-50, 500)):
    """
    Plot raw step-function bid curves (price vs cumulative quantity) for the
    top unit per technology. Two figures per market:

      Figure 1 — curves colored by regime (one panel per tech).
      Figure 2 — curves colored by hour-class (critical vs flat) at the
                 focus regime (default DA15/ID15).

    Each step function is one (date, period, unit) bid; cumulative quantity
    on x, price on y. Many curves are overlaid with low alpha to reveal the
    distribution of bid shapes.
    """
    # 1. Pick the top unit per tech (by total volume in the panel)
    rep = {}
    for tech in TECHS_PRIMARY:
        top = panel[panel['tech'] == tech].groupby('unit_code')['q_total'].sum().nlargest(1)
        if len(top) > 0: rep[tech] = top.index[0]
    if not rep:
        print('No representative units found.')
        return

    # 2. Sample (date, period) per (regime, tech) without replacement
    samples_per_regime = {}
    for regime in regimes:
        rows = []
        for tech, unit in rep.items():
            sub = panel[(panel['regime'] == regime) & (panel['unit_code'] == unit)]
            if len(sub) == 0: continue
            sample = sub.sample(min(n_curves_per_regime, len(sub)), random_state=42)
            rows.append(sample[['date', 'period', 'unit_code']].assign(tech=tech))
        if rows:
            samples_per_regime[regime] = pd.concat(rows, ignore_index=True)
    if not samples_per_regime:
        print('No samples after filtering.')
        return
    all_samples = pd.concat(samples_per_regime.values(), ignore_index=True)

    # 3. Fetch tranches from DuckDB for the sampled (date) × representative units
    fetch = _fetch_tranches_da if market == 'DA' else _fetch_tranches_ida
    tranches = fetch(list(rep.values()), all_samples['date'].unique())
    if len(tranches) == 0:
        print('No tranches returned by DuckDB query.')
        return
    tranches['date'] = pd.to_datetime(tranches['date'])
    tranches['regime'] = tranches['date'].apply(regime_of)
    cutoff = MTU15_DA_DATE if market == 'DA' else ISP15_END
    tranches['hour'] = np.where(
        tranches['date'] >= cutoff, (tranches['period'] - 1) // 4, (tranches['period'] - 1)
    ).clip(0, 23)

    # 4. Restrict to the actually-sampled (date, period, unit) tuples
    keys = set(zip(all_samples['date'].dt.normalize(), all_samples['period'], all_samples['unit_code']))
    tranches['key'] = list(zip(tranches['date'], tranches['period'], tranches['unit_code']))
    tranches = tranches[tranches['key'].isin(keys)].drop(columns='key')

    techs = list(rep.keys())
    n_t = len(techs)
    group_cols = ['date', 'session_number', 'period'] if market == 'IDA' else ['date', 'period']

    # ---- Figure 1: curves colored by regime, one panel per tech ----
    cmap = plt.get_cmap('viridis')
    color_regime = {r: cmap(i / max(1, len(regimes) - 1)) for i, r in enumerate(regimes)}
    fig, axes = plt.subplots(1, n_t, figsize=(3.8 * n_t, 4.2))
    if n_t == 1: axes = [axes]
    for ax, tech in zip(axes, techs):
        unit = rep[tech]
        sub = tranches[tranches['unit_code'] == unit]
        for regime in regimes:
            reg_sub = sub[sub['regime'] == regime]
            if len(reg_sub) == 0: continue
            for _, ladder in reg_sub.groupby(group_cols):
                ladder = ladder.sort_values('price')
                cum = np.concatenate(([0], ladder['qty'].cumsum().values))
                pri = np.concatenate(([ladder['price'].iloc[0]], ladder['price'].values))
                ax.step(cum, pri, where='post', alpha=0.25,
                        color=color_regime[regime], linewidth=0.7)
        for r in regimes:
            ax.plot([], [], color=color_regime[r], label=r, linewidth=2)
        ax.legend(fontsize=7, loc='upper left')
        ax.set_title(f'{tech}: {unit}', fontsize=10)
        ax.set_xlabel('cum qty (MWh)')
        ax.set_ylabel('price (€/MWh)')
        ax.set_ylim(y_clip)
        ax.grid(alpha=0.3)
    plt.suptitle(f'{market} bid curves — top unit per tech, colored by regime', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_11_curves_by_regime.png', dpi=110, bbox_inches='tight')
    plt.show()

    # ---- Figure 2: critical vs flat at focus_regime, one panel per tech ----
    if focus_regime not in regimes:
        print(f'focus_regime {focus_regime!r} not in regimes; skipping critical/flat curves.')
        return
    color_class = {'critical': 'tab:red', 'flat': 'tab:blue'}
    fig, axes = plt.subplots(1, n_t, figsize=(3.8 * n_t, 4.2))
    if n_t == 1: axes = [axes]
    for ax, tech in zip(axes, techs):
        unit = rep[tech]
        sub = tranches[(tranches['unit_code'] == unit) & (tranches['regime'] == focus_regime)].copy()
        sub['hour_class'] = np.where(sub['hour'].isin(CRITICAL_HOURS), 'critical',
                                np.where(sub['hour'].isin(FLAT_HOURS), 'flat', 'other'))
        sub = sub[sub['hour_class'].isin(['critical', 'flat'])]
        for hc in ['critical', 'flat']:
            hc_sub = sub[sub['hour_class'] == hc]
            if len(hc_sub) == 0: continue
            for _, ladder in hc_sub.groupby(group_cols):
                ladder = ladder.sort_values('price')
                cum = np.concatenate(([0], ladder['qty'].cumsum().values))
                pri = np.concatenate(([ladder['price'].iloc[0]], ladder['price'].values))
                ax.step(cum, pri, where='post', alpha=0.4,
                        color=color_class[hc], linewidth=0.8)
        for hc in ['critical', 'flat']:
            ax.plot([], [], color=color_class[hc], label=hc, linewidth=2)
        ax.legend(fontsize=8)
        ax.set_title(f'{tech}: {unit}', fontsize=10)
        ax.set_xlabel('cum qty (MWh)')
        ax.set_ylabel('price (€/MWh)')
        ax.set_ylim(y_clip)
        ax.grid(alpha=0.3)
    plt.suptitle(f'{market} bid curves @ {focus_regime} — critical (red) vs flat (blue) hours, top unit per tech',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_12_curves_crit_flat.png', dpi=110, bbox_inches='tight')
    plt.show()

print('section_bid_curves defined.')


# --- Additional bid-curve views (no aggregation in the curves themselves) ---

def section_bid_curves_by_firm(panel, market, focus_regime='DA15/ID15',
                                 hour_class='critical', n_curves_per_firm=20,
                                 y_clip=(-50, 500)):
    """
    Bid curves at focus_regime + selected hour_class, one curve per (date, period, unit)
    bid. For each dominant firm, take the top CCGT unit and sample bids; plot all
    curves on a single panel colored by firm. Reveals cross-firm bid-shape differences
    at the same regime + hour-class.
    """
    firm_units = {}
    for firm in DOMINANT:
        top = (panel[(panel['tech'] == 'CCGT') & (panel['firm'] == firm)]
               .groupby('unit_code')['q_total'].sum().nlargest(1))
        if len(top) > 0: firm_units[firm] = top.index[0]
    if not firm_units:
        print('No CCGT units found per dominant firm.')
        return

    hours_set = CRITICAL_HOURS if hour_class == 'critical' else FLAT_HOURS
    samples = []
    for firm, unit in firm_units.items():
        sub = panel[(panel['regime'] == focus_regime) &
                    (panel['unit_code'] == unit) &
                    (panel['hour'].isin(hours_set))]
        if len(sub) == 0: continue
        s = sub.sample(min(n_curves_per_firm, len(sub)), random_state=42)
        samples.append(s[['date', 'period', 'unit_code']].assign(firm=firm))
    if not samples:
        print(f'No samples for {focus_regime} + {hour_class}.')
        return
    sample_df = pd.concat(samples, ignore_index=True)

    fetch = _fetch_tranches_da if market == 'DA' else _fetch_tranches_ida
    tranches = fetch(list(firm_units.values()), sample_df['date'].unique())
    if len(tranches) == 0:
        print('No tranches returned.')
        return
    tranches['date'] = pd.to_datetime(tranches['date'])
    keys = set(zip(sample_df['date'].dt.normalize(), sample_df['period'], sample_df['unit_code']))
    tranches['key'] = list(zip(tranches['date'], tranches['period'], tranches['unit_code']))
    tranches = tranches[tranches['key'].isin(keys)].drop(columns='key')

    cmap = plt.get_cmap('tab10')
    color_firm = {f: cmap(i) for i, f in enumerate(firm_units.keys())}
    group_cols = ['date', 'session_number', 'period'] if market == 'IDA' else ['date', 'period']

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for firm, unit in firm_units.items():
        sub = tranches[tranches['unit_code'] == unit]
        for _, ladder in sub.groupby(group_cols):
            ladder = ladder.sort_values('price')
            cum = np.concatenate(([0], ladder['qty'].cumsum().values))
            pri = np.concatenate(([ladder['price'].iloc[0]], ladder['price'].values))
            ax.step(cum, pri, where='post', alpha=0.30,
                    color=color_firm[firm], linewidth=0.8)
    for f, u in firm_units.items():
        ax.plot([], [], color=color_firm[f], label=f'{f}: {u}', linewidth=2)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlabel('cum qty (MWh)'); ax.set_ylabel('price (€/MWh)')
    ax.set_ylim(y_clip); ax.grid(alpha=0.3)
    ax.set_title(f'{market} bid curves @ {focus_regime}, {hour_class} hours — top CCGT per dominant firm', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_13_curves_by_firm.png', dpi=110, bbox_inches='tight')
    plt.show()


def section_bid_curves_by_hour(panel, market, focus_regime='DA15/ID15',
                                 n_curves_per_hour=4, y_clip=(-50, 500)):
    """
    Bid curves at focus_regime, for the top unit per tech, colored by hour-of-day
    (0-23) using a continuous colormap. Reveals how a unit's bid SHAPE varies
    across the day at the same regime — finer than the critical/flat dichotomy.
    """
    rep = {}
    for tech in TECHS_PRIMARY:
        top = panel[panel['tech'] == tech].groupby('unit_code')['q_total'].sum().nlargest(1)
        if len(top) > 0: rep[tech] = top.index[0]
    if not rep:
        print('No representative units.')
        return

    samples = []
    for tech, unit in rep.items():
        sub = panel[(panel['regime'] == focus_regime) & (panel['unit_code'] == unit)]
        if len(sub) == 0: continue
        for h in range(24):
            sub_h = sub[sub['hour'] == h]
            if len(sub_h) == 0: continue
            s = sub_h.sample(min(n_curves_per_hour, len(sub_h)), random_state=42)
            samples.append(s[['date', 'period', 'unit_code']].assign(tech=tech, hour=h))
    if not samples:
        print('No samples.')
        return
    sample_df = pd.concat(samples, ignore_index=True)

    fetch = _fetch_tranches_da if market == 'DA' else _fetch_tranches_ida
    tranches = fetch(list(rep.values()), sample_df['date'].unique())
    if len(tranches) == 0:
        print('No tranches returned.')
        return
    tranches['date'] = pd.to_datetime(tranches['date'])
    cutoff = MTU15_DA_DATE if market == 'DA' else ISP15_END
    tranches['hour'] = np.where(
        tranches['date'] >= cutoff, (tranches['period'] - 1) // 4, (tranches['period'] - 1)
    ).clip(0, 23)
    keys = set(zip(sample_df['date'].dt.normalize(), sample_df['period'], sample_df['unit_code']))
    tranches['key'] = list(zip(tranches['date'], tranches['period'], tranches['unit_code']))
    tranches = tranches[tranches['key'].isin(keys)].drop(columns='key')

    cmap = plt.get_cmap('twilight_shifted', 24)  # cyclic colormap for hours
    techs = list(rep.keys())
    n_t = len(techs)
    group_cols = ['date', 'session_number', 'period'] if market == 'IDA' else ['date', 'period']

    fig, axes = plt.subplots(1, n_t, figsize=(3.8 * n_t, 4.5))
    if n_t == 1: axes = [axes]
    for ax, tech in zip(axes, techs):
        unit = rep[tech]
        sub = tranches[(tranches['unit_code'] == unit) & (tranches['regime'] == focus_regime if 'regime' in tranches.columns else True)]
        # regime column may not exist on tranches; recompute
        sub = tranches[tranches['unit_code'] == unit].copy()
        sub['regime'] = sub['date'].apply(regime_of)
        sub = sub[sub['regime'] == focus_regime]
        for h in range(24):
            h_sub = sub[sub['hour'] == h]
            if len(h_sub) == 0: continue
            for _, ladder in h_sub.groupby(group_cols):
                ladder = ladder.sort_values('price')
                cum = np.concatenate(([0], ladder['qty'].cumsum().values))
                pri = np.concatenate(([ladder['price'].iloc[0]], ladder['price'].values))
                ax.step(cum, pri, where='post', alpha=0.40,
                        color=cmap(h / 23), linewidth=0.8)
        ax.set_title(f'{tech}: {unit}', fontsize=10)
        ax.set_xlabel('cum qty (MWh)')
        ax.set_ylabel('price (€/MWh)')
        ax.set_ylim(y_clip); ax.grid(alpha=0.3)
    # Single colorbar (shared)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=23))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, location='right', label='hour of day', pad=0.01)
    cbar.set_ticks(range(0, 24, 3))
    plt.suptitle(f'{market} bid curves @ {focus_regime} — top unit per tech, colored by hour-of-day', fontsize=11)
    plt.savefig(OUT / f'bid_atlas_{market}_14_curves_by_hour.png', dpi=110, bbox_inches='tight')
    plt.show()


def section_bid_curves_multi_unit(panel, market, focus_regime='DA15/ID15',
                                    tech='CCGT', top_n_units=5, n_curves_per_unit=15,
                                    y_clip=(-50, 500)):
    """
    Bid curves at focus_regime for the top-N units of a chosen technology
    (default CCGT). All curves on a single panel; one color per unit. Reveals
    cross-unit shape variation within a single tech without any aggregation.
    """
    units = (panel[panel['tech'] == tech]
             .groupby(['unit_code', 'firm'])['q_total'].sum()
             .reset_index().nlargest(top_n_units, 'q_total'))
    if len(units) == 0:
        print(f'No {tech} units found.')
        return

    samples = []
    for _, row in units.iterrows():
        sub = panel[(panel['regime'] == focus_regime) & (panel['unit_code'] == row['unit_code'])]
        if len(sub) == 0: continue
        s = sub.sample(min(n_curves_per_unit, len(sub)), random_state=42)
        samples.append(s[['date', 'period', 'unit_code']].assign(firm=row['firm']))
    if not samples:
        print('No samples.')
        return
    sample_df = pd.concat(samples, ignore_index=True)

    fetch = _fetch_tranches_da if market == 'DA' else _fetch_tranches_ida
    tranches = fetch(units['unit_code'].tolist(), sample_df['date'].unique())
    if len(tranches) == 0:
        print('No tranches returned.')
        return
    tranches['date'] = pd.to_datetime(tranches['date'])
    keys = set(zip(sample_df['date'].dt.normalize(), sample_df['period'], sample_df['unit_code']))
    tranches['key'] = list(zip(tranches['date'], tranches['period'], tranches['unit_code']))
    tranches = tranches[tranches['key'].isin(keys)].drop(columns='key')

    cmap = plt.get_cmap('tab10')
    unit_color = {u: cmap(i) for i, u in enumerate(units['unit_code'].tolist())}
    group_cols = ['date', 'session_number', 'period'] if market == 'IDA' else ['date', 'period']

    fig, ax = plt.subplots(figsize=(10, 5.5))
    for _, row in units.iterrows():
        unit = row['unit_code']
        firm = row['firm']
        sub = tranches[tranches['unit_code'] == unit]
        for _, ladder in sub.groupby(group_cols):
            ladder = ladder.sort_values('price')
            cum = np.concatenate(([0], ladder['qty'].cumsum().values))
            pri = np.concatenate(([ladder['price'].iloc[0]], ladder['price'].values))
            ax.step(cum, pri, where='post', alpha=0.25,
                    color=unit_color[unit], linewidth=0.7)
    for _, row in units.iterrows():
        ax.plot([], [], color=unit_color[row['unit_code']],
                label=f"{row['unit_code']} ({row['firm']})", linewidth=2)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlabel('cum qty (MWh)'); ax.set_ylabel('price (€/MWh)')
    ax.set_ylim(y_clip); ax.grid(alpha=0.3)
    ax.set_title(f'{market} bid curves @ {focus_regime} — top-{top_n_units} {tech} units (no aggregation)', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_15_curves_multi_{tech}.png', dpi=110, bbox_inches='tight')
    plt.show()

print('Extended bid-curve helpers defined.')



print('Time-series helpers defined.')


def section_bid_timeseries(panel, market, units=None,
                            measures=None, window_days=7):
    """
    Layered time-series of bid measures, for the top unit per tech.

    Each panel (one per tech-measure combo) shows:
      - very light scatter of every individual bid (alpha=0.02)
      - daily median line (thin) for critical and flat hours separately
      - rolling-window median (window_days, default 7) overlaid (thick line)

    Critical hours in red, flat hours in blue. Reform dates marked with
    vertical grey lines. No regime-level aggregation; the only aggregation
    is across-period within-day, which is the minimal time bin for
    intra-day-variant outcomes.
    """
    if units is None:
        units = {}
        for tech in TECHS_PRIMARY:
            sub_t = panel[panel['tech'] == tech]
            if len(sub_t) == 0: continue
            top = sub_t.groupby('unit_code')['q_total'].sum().nlargest(1)
            if len(top) > 0: units[tech] = top.index[0]
    if measures is None:
        measures = ['n_tranches', 'markup_span_pct', 'top_premium_pct', 'floor_discount_pct']

    techs = list(units.keys())
    n_t, n_m = len(techs), len(measures)
    if n_t == 0:
        print('No units.')
        return

    fig, axes = plt.subplots(n_m, n_t, figsize=(3.0 * n_t, 2.0 * n_m),
                              sharex=True, squeeze=False)
    reform_dates = [PRE_IDA_END, THREE_SESS_END, ISP15_END, BLACKOUT, MTU15_DA_DATE]
    color_class = {'critical': 'tab:red', 'flat': 'tab:blue'}

    for j, tech in enumerate(techs):
        unit = units[tech]
        sub = panel[panel['unit_code'] == unit].copy()
        sub['hour_class'] = np.where(sub['hour'].isin(CRITICAL_HOURS), 'critical',
                                np.where(sub['hour'].isin(FLAT_HOURS), 'flat', 'other'))

        for i, measure in enumerate(measures):
            ax = axes[i, j]
            for hc in ['critical', 'flat']:
                hcsub = sub[sub['hour_class'] == hc]
                if len(hcsub) == 0: continue
                # Light scatter (the "fog")
                ax.scatter(hcsub['date'], hcsub[measure],
                            alpha=0.02, s=1, color=color_class[hc],
                            edgecolors='none')
                # Daily median
                daily = hcsub.groupby(hcsub['date'].dt.date)[measure].median()
                daily.index = pd.to_datetime(daily.index)
                ax.plot(daily.index, daily.values, color=color_class[hc],
                         alpha=0.4, lw=0.5)
                # Rolling 7-day median (thick line)
                rolling = daily.rolling(f'{window_days}D', min_periods=1).median()
                ax.plot(rolling.index, rolling.values, color=color_class[hc],
                         alpha=0.95, lw=1.6, label=hc if (i == 0 and j == 0) else None)
            for d in reform_dates:
                ax.axvline(d, color='grey', alpha=0.5, lw=0.7, ls='--')
            if i == 0:
                ax.set_title(f'{tech}: {unit}', fontsize=9)
            if j == 0:
                ax.set_ylabel(measure, fontsize=9)
            ax.grid(alpha=0.3); ax.tick_params(labelsize=7)

    if n_t > 0 and n_m > 0:
        axes[0, 0].legend(fontsize=7, loc='upper left')
    plt.suptitle(f'{market} bid measures over time — top unit per tech, '
                 f'critical (red) vs flat (blue) hours, {window_days}-day rolling median', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_16_timeseries.png', dpi=110, bbox_inches='tight')
    plt.show()


def section_bid_timeseries_by_firm(panel, market, tech='CCGT',
                                     measures=None, window_days=7):
    """
    Layered time-series, top tech-unit per dominant firm. Each panel shows
    the rolling-window median of one bid measure for each firm's top unit
    in critical hours; underlying scatter shown lightly. Reveals when
    different dominant firms diverge in bid behaviour.
    """
    firm_units = {}
    for firm in DOMINANT:
        top = (panel[(panel['tech'] == tech) & (panel['firm'] == firm)]
               .groupby('unit_code')['q_total'].sum().nlargest(1))
        if len(top) > 0: firm_units[firm] = top.index[0]
    if not firm_units:
        print(f'No {tech} units per dominant firm.')
        return
    if measures is None:
        measures = ['n_tranches', 'markup_span_pct', 'top_premium_pct', 'floor_discount_pct']

    cmap = plt.get_cmap('tab10')
    color_firm = {f: cmap(i) for i, f in enumerate(firm_units.keys())}
    n_m = len(measures)
    fig, axes = plt.subplots(n_m, 1, figsize=(11, 2.0 * n_m), sharex=True, squeeze=False)
    reform_dates = [PRE_IDA_END, THREE_SESS_END, ISP15_END, BLACKOUT, MTU15_DA_DATE]

    for i, measure in enumerate(measures):
        ax = axes[i, 0]
        for firm, unit in firm_units.items():
            sub = panel[(panel['unit_code'] == unit) &
                        (panel['hour'].isin(CRITICAL_HOURS))]
            if len(sub) == 0: continue
            # Daily median
            daily = sub.groupby(sub['date'].dt.date)[measure].median()
            daily.index = pd.to_datetime(daily.index)
            # Rolling
            rolling = daily.rolling(f'{window_days}D', min_periods=1).median()
            ax.plot(rolling.index, rolling.values, color=color_firm[firm],
                     alpha=0.9, lw=1.4, label=f'{firm}: {unit}' if i == 0 else None)
        for d in reform_dates:
            ax.axvline(d, color='grey', alpha=0.5, lw=0.5, ls='--')
        ax.set_ylabel(measure, fontsize=9)
        ax.grid(alpha=0.3); ax.tick_params(labelsize=8)
    axes[0, 0].legend(fontsize=8, loc='upper left')
    plt.suptitle(f'{market} {tech} bid measures (critical hours, {window_days}-day rolling median) — top unit per dominant firm', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT / f'bid_atlas_{market}_17_timeseries_firm_{tech}.png', dpi=110, bbox_inches='tight')
    plt.show()

print('Redesigned time-series helpers defined.')


## Section 1 — Bid panel construction

For each (date, period, unit) we compute five bid-shape measures from the offer-detail tables (DET / IDET) joined to the offer headers (CAB / ICAB). Two panels are built:

- **DA panel** — DET joined to CAB, restricted to post-2025-03-19 because pre-MTU15-IDA DA bid-prices are 0-padded due to a parser artefact.
- **IDA panel** — IDET joined to ICAB. Spans the full reform sequence cleanly.


In [ ]:
# DA bid panel — sell offers post-MTU15-IDA (where DET prices are clean)
da_bids = con.execute(f'''
    WITH cab_latest AS (
      SELECT CAST(date AS DATE) AS d, offer_code, version, unit_code,
             ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), offer_code
                                ORDER BY version DESC) AS rn
      FROM '{CAB}'
      WHERE buy_sell = 'V'
        AND CAST(date AS DATE) >= DATE '2025-03-19'
    ),
    cab_v AS (SELECT * FROM cab_latest WHERE rn = 1),
    det_clean AS (
      SELECT CAST(date AS DATE) AS d, offer_code, version, period,
             price_eur_mwh AS price, quantity_mw AS qty
      FROM '{DET}'
      WHERE quantity_mw IS NOT NULL AND quantity_mw > 0
        AND price_eur_mwh IS NOT NULL
        AND CAST(date AS DATE) >= DATE '2025-03-19'
        AND period BETWEEN 1 AND 96
    )
    SELECT det.d AS date, det.period, cab_v.unit_code,
           unit_map.firm, unit_map.tech, unit_map.firm_class,
           COUNT(DISTINCT det.price) AS n_tranches,
           SUM(det.qty)              AS q_total,
           MIN(det.price)            AS p_min,
           MAX(det.price)            AS p_max,
           SUM(det.price * det.qty) / SUM(det.qty) AS p_avg
    FROM det_clean det
      JOIN cab_v USING (d, offer_code, version)
      JOIN unit_map ON cab_v.unit_code = unit_map.unit_code
    GROUP BY 1, 2, 3, 4, 5, 6
''').df()

da_bids['date']    = pd.to_datetime(da_bids['date'])
da_bids['regime']  = da_bids['date'].apply(regime_of)
da_bids['slope']   = (da_bids['p_max'] - da_bids['p_min']) / da_bids['q_total'].clip(lower=1.0)
da_bids['p_range'] = da_bids['p_max'] - da_bids['p_min']
da_bids['hour']    = np.where(
    da_bids['date'] >= MTU15_DA_DATE,
    (da_bids['period'] - 1) // 4,
    (da_bids['period'] - 1)
).clip(0, 23)
da_bids['quarter'] = np.where(da_bids['date'] >= MTU15_DA_DATE, (da_bids['period'] - 1) % 4, 0)

print(f'DA panel rows (post-2025-03-19): {len(da_bids):,}')
print(f'Distinct units: {da_bids.unit_code.nunique():,}')
print(da_bids.groupby('regime').size().reindex(REGIMES_3_DA))

# Interpretable shape measures (derived from existing columns)
da_bids['top_premium_pct']   = (da_bids['p_max'] - da_bids['p_avg']) / da_bids['p_avg'].abs().clip(lower=1.0)
da_bids['floor_discount_pct'] = (da_bids['p_avg'] - da_bids['p_min']) / da_bids['p_avg'].abs().clip(lower=1.0)
da_bids['markup_span_pct']    = (da_bids['p_max'] - da_bids['p_min']) / da_bids['p_avg'].abs().clip(lower=1.0)
print('DA shape measures: top_premium_pct, floor_discount_pct, markup_span_pct added.')


In [ ]:
# IDA bid panel — sell offers across all regimes
# IDET has unit_code directly; ICAB used only to filter buy_sell='V' on the offer header
ida_bids = con.execute(f'''
    WITH icab_latest AS (
      SELECT CAST(date AS DATE) AS d, session_number, offer_code, version,
             ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), session_number, offer_code
                                ORDER BY version DESC) AS rn
      FROM '{ICAB}' WHERE buy_sell = 'V'
    ),
    icab_v AS (SELECT * FROM icab_latest WHERE rn = 1),
    idet_clean AS (
      SELECT CAST(date AS DATE) AS d, session_number, offer_code, version, period, unit_code,
             price_eur_mwh AS price, quantity_mw AS qty
      FROM '{IDET}'
      WHERE quantity_mw IS NOT NULL AND quantity_mw > 0
        AND price_eur_mwh IS NOT NULL
        AND period BETWEEN 1 AND 96
    )
    SELECT idet.d AS date, idet.session_number, idet.period, idet.unit_code,
           unit_map.firm, unit_map.tech, unit_map.firm_class,
           COUNT(DISTINCT idet.price) AS n_tranches,
           SUM(idet.qty)              AS q_total,
           MIN(idet.price)            AS p_min,
           MAX(idet.price)            AS p_max,
           SUM(idet.price * idet.qty) / SUM(idet.qty) AS p_avg
    FROM idet_clean idet
      JOIN icab_v USING (d, session_number, offer_code, version)
      JOIN unit_map ON idet.unit_code = unit_map.unit_code
    GROUP BY 1, 2, 3, 4, 5, 6, 7
''').df()

ida_bids['date']    = pd.to_datetime(ida_bids['date'])
ida_bids['regime']  = ida_bids['date'].apply(regime_of)
ida_bids['slope']   = (ida_bids['p_max'] - ida_bids['p_min']) / ida_bids['q_total'].clip(lower=1.0)
ida_bids['p_range'] = ida_bids['p_max'] - ida_bids['p_min']
ida_bids['hour']    = np.where(
    ida_bids['date'] >= ISP15_END,
    (ida_bids['period'] - 1) // 4,
    (ida_bids['period'] - 1)
).clip(0, 23)
ida_bids['quarter'] = np.where(ida_bids['date'] >= ISP15_END, (ida_bids['period'] - 1) % 4, 0)

print(f'IDA panel rows: {len(ida_bids):,}')
print(f'Distinct units: {ida_bids.unit_code.nunique():,}')
print(ida_bids.groupby('regime').size().reindex(REGIMES_6))

# Interpretable shape measures (derived from existing columns)
ida_bids['top_premium_pct']   = (ida_bids['p_max'] - ida_bids['p_avg']) / ida_bids['p_avg'].abs().clip(lower=1.0)
ida_bids['floor_discount_pct'] = (ida_bids['p_avg'] - ida_bids['p_min']) / ida_bids['p_avg'].abs().clip(lower=1.0)
ida_bids['markup_span_pct']    = (ida_bids['p_max'] - ida_bids['p_min']) / ida_bids['p_avg'].abs().clip(lower=1.0)
print('IDA shape measures: top_premium_pct, floor_discount_pct, markup_span_pct added.')


# Part A — Day-Ahead market (DA)

Restricted to post-2025-03-19 (three regimes: DA60/ID15-prebo, DA60/ID15-reforz, DA15/ID15) because DA bid-prices pre-MTU15-IDA are 0-padded due to a parser artefact.


### A.1 Across regimes (overall)

In [ ]:
section_overall(da_bids, REGIMES_3_DA, 'DA')

### A.2 By technology × regime

In [ ]:
section_by_tech(da_bids, REGIMES_3_DA, 'DA')

### A.3 By firm × regime

In [ ]:
section_by_firm(da_bids, REGIMES_3_DA, 'DA')

### A.4 Hour-of-day × regime

In [ ]:
section_hour_heatmap(da_bids, REGIMES_3_DA, 'DA')

### A.5 Critical-flat differential by tech × regime

In [ ]:
section_crit_flat(da_bids, REGIMES_3_DA, 'DA')

### A.6 Seasonality (month × regime)

In [ ]:
section_seasonality(da_bids, REGIMES_3_DA, 'DA')

### A.7 Plant-level drilldown

In [ ]:
section_plant_level(da_bids, REGIMES_3_DA, 'DA')

### A.8 Within-hour granularity exploitation (post-MTU15-DA)

Mechanical-repeat indicator: for each (date, hour, unit) post-2025-10-01 (when DA goes 15-min), check whether all four DA quarter-bids are identical. Critical hours (where the within-market granularity mechanism predicts strategic differentiation) should show lower mechanical-repeat rates than flat hours.


In [ ]:
section_within_hour(da_bids, MTU15_DA_DATE, 'DA')

### A.9 Raw bid curves (price-quantity step functions)

Two visualisations of the actual bid LADDERS for the top unit per technology, post-2025-03-19:

- **Figure 1**: bid curves colored by regime — shows how the same unit's bid SHAPE evolves across reform regimes.
- **Figure 2**: bid curves at DA15/ID15, colored by hour-class — shows the within-day differential at the unit level (critical hours red, flat hours blue).

Each step function is one (date, period, unit) bid: cumulative quantity on x, price on y. Many curves are overlaid with low alpha to reveal the distribution of bid shapes rather than a single bid.


In [ ]:
section_bid_curves(da_bids, REGIMES_3_DA, 'DA', focus_regime='DA15/ID15')

### A.10 Bid curves by dominant firm (no aggregation)

At DA15/ID15 + critical hours, plot bid curves for the top CCGT of each dominant firm. All curves on a single panel, colored by firm. Lets you see cross-firm bid-shape variation directly without medians or means.


In [ ]:
section_bid_curves_by_firm(da_bids, 'DA', focus_regime='DA15/ID15', hour_class='critical')

### A.11 Bid curves by hour-of-day (no aggregation)

At DA15/ID15, plot bid curves for the top unit per technology, colored by hour-of-day (cyclic colormap). Reveals the within-day shape continuum — finer than the critical/flat dichotomy.


In [ ]:
section_bid_curves_by_hour(da_bids, 'DA', focus_regime='DA15/ID15')

### A.12 Bid curves across multiple units of the same tech (no aggregation)

At DA15/ID15, plot bid curves for the top-5 CCGT units, all on a single panel, one color per unit. Reveals cross-unit shape variation within CCGT directly.


In [ ]:
section_bid_curves_multi_unit(da_bids, 'DA', focus_regime='DA15/ID15', tech='CCGT', top_n_units=5)
section_bid_curves_multi_unit(da_bids, 'DA', focus_regime='DA15/ID15', tech='hydro', top_n_units=5)

### A.13 Bid measures over time — no time aggregation

Time-series scatter of bid-shape measures: every (date, period, unit) bid is one point on the (date, measure) plane. No medians, no percentiles, no regime bins. The full temporal distribution of bids is visible directly. Reform dates are marked with vertical red lines.

This is the most disaggregated view of how bid behaviour evolves over time at the unit level.


In [ ]:
section_bid_timeseries(da_bids, 'DA')
section_bid_timeseries_by_firm(da_bids, 'DA', tech='CCGT')

# Part B — Intraday Auctions market (IDA)

Spans the full reform sequence (six regimes). IDET prices are clean throughout, so the IDA panel provides the cross-regime view that DA cannot.


### B.1 Across regimes (overall)

In [ ]:
section_overall(ida_bids, REGIMES_6, 'IDA')

### B.2 By technology × regime

In [ ]:
section_by_tech(ida_bids, REGIMES_6, 'IDA')

### B.3 By firm × regime

In [ ]:
section_by_firm(ida_bids, REGIMES_6, 'IDA')

### B.4 Hour-of-day × regime

In [ ]:
section_hour_heatmap(ida_bids, REGIMES_6, 'IDA')

### B.5 Critical-flat differential by tech × regime

In [ ]:
section_crit_flat(ida_bids, REGIMES_6, 'IDA')

### B.6 Seasonality (month × regime)

In [ ]:
section_seasonality(ida_bids, REGIMES_6, 'IDA')

### B.7 Plant-level drilldown

In [ ]:
section_plant_level(ida_bids, REGIMES_6, 'IDA')

### B.8 Within-hour granularity exploitation (post-MTU15-IDA)

Mechanical-repeat indicator: for each (date, hour, unit) post-2025-03-19 (when IDA goes 15-min), check whether all four IDA quarter-bids are identical. Same logic as A.8 but on the IDA panel.


In [ ]:
section_within_hour(ida_bids, ISP15_END, 'IDA')

### B.9 Raw bid curves (price-quantity step functions)

Same two visualisations on the IDA panel: curves colored by regime, then curves at DA15/ID15 colored by hour-class. With six regimes available, the cross-regime evolution is more granular than on DA.


In [ ]:
section_bid_curves(ida_bids, REGIMES_6, 'IDA', focus_regime='DA15/ID15')

### B.10 Bid curves by dominant firm (no aggregation)

Same as §A.10 but on the IDA panel.


In [ ]:
section_bid_curves_by_firm(ida_bids, 'IDA', focus_regime='DA15/ID15', hour_class='critical')

### B.11 Bid curves by hour-of-day (no aggregation)

Same as §A.11 but on the IDA panel.


In [ ]:
section_bid_curves_by_hour(ida_bids, 'IDA', focus_regime='DA15/ID15')

### B.12 Bid curves across multiple units of the same tech (no aggregation)

Same as §A.12 but on the IDA panel.


In [ ]:
section_bid_curves_multi_unit(ida_bids, 'IDA', focus_regime='DA15/ID15', tech='CCGT', top_n_units=5)
section_bid_curves_multi_unit(ida_bids, 'IDA', focus_regime='DA15/ID15', tech='hydro', top_n_units=5)

### B.13 Bid measures over time — no time aggregation

Same time-series scatter as §A.13 but on the IDA panel — covers the full reform sequence (2018 onward) so the temporal evolution is visible across all regimes.


In [ ]:
section_bid_timeseries(ida_bids, 'IDA')
section_bid_timeseries_by_firm(ida_bids, 'IDA', tech='CCGT')

## Synthesis

Both auction markets, eight analyses each, identical code:

- **Across regimes (§A.1, §B.1)** — overall bid-shape level shifts across the reform sequence.
- **By technology (§A.2, §B.2)** — CCGT and hydro should show richer bid-shape variation than wind/solar/nuclear under the within-market granularity mechanism.
- **By firm (§A.3, §B.3)** — dominant vs fringe gap, and per-dominant heterogeneity. Connects to the fringe-placebo result (B13).
- **Hour-of-day (§A.4, §B.4)** — heatmaps with critical hours marked red, flat hours blue. Reveals where in the day bid-shape varies systematically.
- **Critical-flat differential (§A.5, §B.5)** — the headline mechanism. Predicted positive and growing at MTU15-DA for CCGT/hydro; ~zero for wind/solar/nuclear.
- **Seasonality (§A.6, §B.6)** — month × regime heatmaps for dispatchable techs.
- **Plant-level (§A.7, §B.7)** — top units per tech, per-unit bid measures.
- **Within-hour granularity exploitation (§A.8, §B.8)** — the mechanical-repeat rate is the cleanest descriptive signature of operators using or not using the four-quarter granularity.

DA covers three regimes (post-2025-03-19); IDA covers all six. Comparing the same firm × tech × hour stratification across markets reveals whether DA and IDA show distinct or aligned strategic signatures.

These descriptive facts feed into:

- `notebooks/memos/_modelling_track.md` § X (within-day DiD identification design).
- `notebooks/memos/_within_market_granularity_model.md` (stylised IO model that should formalise these patterns).
- `CLAIMS_LEDGER.md` rows B14 (bid-shape rich-ladder pattern) and B12/B13 (within-day DiD headline + fringe placebo).

**Caveats.** Pre-2025-03-19 DA bid prices are 0-padded due to a parser artefact; the DA panel is restricted accordingly. Within-hour granularity exploitation analyses operate only on the post-MTU15 portion of each market's panel.
